# TUTOR-60 · 2 — Planning the experiment

Notebook 1 ended with a number: resolving **which allocation to fund** is worth about
2.3 million dollars a year, seventeen times what resolving go/no-go is worth. This
notebook turns that into a trial — and every choice in it is made against that number
rather than against a convention.

Four decisions, in the order they constrain each other:

| | decides | reaches into |
|---|---|---|
| **1** | randomize students or schools? | `design.ClusterDesign`, `design.design_effect`, `design.clusters_needed` |
| **2** | which outcome column? | `design.mde`, `design.sample_size` |
| **3** | which allocations, and how many distinct ones? | `surface.optimal_exchange`, `surface.design_matrix`, `surface.d_criterion` |
| **4** | how many schools? | `design.evoi_gaussian`, `design.mid_horizon_factor` |

The fourth is the one everybody asks first, and it cannot be answered until the other
three are settled, because each of them changes the answer by a factor of two or more.

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import tutoring as T
from axiom.core import D, Unit, is_failure
from axiom.design import (
    ClusterDesign, DecisionSpec, MDE, SampleSize, cluster_mde, clusters_needed, design_effect,
    discount_weights, effective_sample_size, evoi_gaussian, evpi_gaussian, mde, mid_horizon_factor,
    sample_size,
)
from axiom.surface import Surface, d_criterion, design_matrix, optimal_exchange

# The prior on the decision quantity, as notebook 1 left it.
PRIOR_MEAN, PRIOR_SD = 6738.0, 10414.0
dose_decision = DecisionSpec(name="prefer_the_low_allocation", threshold=0.0,
                             value_per_outcome_unit=T.VALUE_PER_POINT, numeraire="USD")
evpi = evpi_gaussian(dose_decision, PRIOR_MEAN, PRIOR_SD)
print(f"EVPI on the dose decision: {evpi:,.0f} USD per program year")

# A twelve-school feasibility pilot ran last spring; it is where the variance numbers
# below come from, and it is all anybody has.
pilot = T.run(T.snap([(600.0, 0.0), (600.0, 45.0), (1800.0, 0.0), (1800.0, 45.0)] * 3), seed=99)
print(f"feasibility pilot: {pilot.n_schools} schools, {pilot.students.shape[0]:,} eligible students")

## 1 · Students or schools

A tutor works with students from several classrooms; a school that runs the program
changes how its teachers use the rest of the block. Assigning individual students would
put treated and untreated children in the same room and the same intervention. So the
unit of assignment is the **school**, and the trial pays for that in effective sample
size.

In [ ]:
school = Unit(name="school", dimension=D.entity, kind="cluster")
student_sd = float(pilot.students["gain"].std())
icc = pilot.icc("gain")
cluster = ClusterDesign(unit=school, n_clusters=120, cluster_size=50, icc=icc, allocation=0.5)

print(f"intraclass correlation (gain)     {icc:.3f}")
print(f"design effect at 50 per school    {design_effect(50, icc):.2f}")
print(f"effective n from 120x50 students  {effective_sample_size(120, 50, icc):,.0f} (nominal 6,000)")

individual: SampleSize = sample_size(effect=1.5, sd=student_sd, power=0.8)
clustered = clusters_needed(effect=1.5, sd=student_sd, cluster_size=50, icc=icc)
assert isinstance(clustered, SampleSize)
print(f"\nto detect 1.5 {T.OUTCOME_UNIT} at 80% power:")
print(f"  individually randomized  {individual.n:>6,d} students")
print(f"  school randomized        {clustered.n * 50:>6,d} students in {clustered.n} schools "
      f"({clustered.n * 50 / individual.n:.1f}x)")

In [ ]:
counts = np.arange(20, 401, 10)
fig = T.figure("What assigning schools instead of students costs",
               "schools in the trial", f"minimum detectable effect ({T.OUTCOME_UNIT})")
fig.add_trace(go.Scatter(
    x=counts, y=[cluster_mde(cluster.model_copy(update={"n_clusters": int(n)}), sd=student_sd).effect for n in counts],
    name="school randomized, gain", line={"color": T.DECISION_COLOR, "width": 3}))
fig.add_trace(go.Scatter(
    x=counts, y=[mde(int(n) * 50, sd=student_sd, power=0.8).effect for n in counts],
    name="if students could be randomized", line={"color": T.TRUTH_COLOR, "width": 2, "dash": "dash"}))
fig.add_hline(y=1.5, line={"dash": "dot", "color": T.ACCENT},
              annotation_text="1.5 points — the smallest difference worth acting on")
fig.show()

The gap between the two lines is the interference assumption made honest: the same 1.5
points that 6,700 individually randomized children would detect takes 29,000 children in
580 schools once they are assigned in clusters — **4.3 times as many**, and 580 schools do
not exist. On the raw gain scale this trial is not feasible at all.

## 2 · The outcome column is worth more than the trial size

Every school in the state was measured last year too. Most of what makes one school gain
more than another is *persistent* — it shows up in both years — so the difference between
this year's gain and the same school's prior-year gain carries the treatment effect and
almost none of the school. That is one column of arithmetic on data the state already
owns.

In [ ]:
rows = []
for column in ("gain", "growth"):
    school_sd = float(pilot.school_means(column)["outcome"].std())
    rows.append({
        "outcome": column,
        "student sd": float(pilot.students[column].std()),
        "icc": pilot.icc(column),
        "school-level sd": school_sd,
        "MDE at 120 schools": mde(120, sd=school_sd, power=0.8).effect,
        "schools for 1.5 points": sample_size(effect=1.5, sd=school_sd, power=0.8).n,
    })
table = pd.DataFrame(rows).set_index("outcome")
print(table.round(3).to_string())
print(f"\nthe prior year is worth {table.loc['gain', 'schools for 1.5 points'] - table.loc['growth', 'schools for 1.5 points']:,.0f} schools, "
      f"and there is no invoice for it")

**Five hundred schools versus a hundred and fifty.** The state has 240 in the tier, so on the gain scale the trial
cannot be run at all and on the growth scale it can. Nothing about the design, the
sample, or the budget changed — only which column the analysis reads. The intraclass
correlation halves for the same reason: most of what students in a school share is the
part the prior year already measured.

From here on the outcome is `growth`. Note what this has done to the size question: it is
now a question about 150 schools rather than 500, which is the range the state can
actually argue about.

## 3 · Which allocations

The decision needs the **shape** of a curve, not one contrast, so the design question is
where to put 120 schools in the two-dimensional dose region. Both response families the
analysis will consider are linear in their parameters, which means the precision of every
candidate design can be computed exactly, before any data exists, from
`surface.design_matrix`.

The trap is which model to compute it against. A design chosen to be optimal for the
model you hope is right cannot tell you whether it is.

In [ ]:
surface = Surface(T.planning_spec())
linear = [name for name, role in surface.roles.items() if role == "linear"]

# One row per school, because one school holds one allocation for a whole year: its six
# benchmark readings are six looks at the same number. SIGMA is the school-level residual
# sd of growth, straight off the pilot -- the same number section 2 sized the trial with.
SIGMA = float(pilot.school_means("growth")["outcome"].std())
print(f"school-level residual sd from the pilot: {SIGMA:.2f} {T.OUTCOME_UNIT}")

def _dose(points):
    return {"tutoring": np.array([a for a, _ in points]), "messaging": np.array([b for _, b in points])}

def matrix(points, spec_):
    surf = Surface(spec_)
    cols = [name for name, role in surf.roles.items() if role == "linear"]
    return design_matrix(surf.model, cols, _dose(points), {})

def curve_se(points, spec_, grid, *, sigma=SIGMA):
    """Exact posterior sd of the fitted response at each grid dose, from the design alone."""
    X = matrix(points, spec_).X
    information = X.T @ X
    if np.linalg.cond(information) > 1e12:
        return np.full(len(grid), np.nan)
    covariance = np.linalg.inv(information) * sigma**2
    contrast = matrix([(d, T.MESSAGING_MAX) for d in grid], spec_).X - matrix(
        [(0.0, 0.0) for _ in grid], spec_).X
    return np.sqrt(np.clip(np.einsum("ij,jk,ik->i", contrast, covariance, contrast), 0.0, None))

from axiom.surface import SplineKernel

def spline_spec(knots):
    return T.spec({"tutoring": SplineKernel(reference_dose=T.TUTORING_MAX, amplitude_scale=8.0, knots=knots),
                   "messaging": SplineKernel(reference_dose=T.MESSAGING_MAX, amplitude_scale=2.0,
                                             knots=T.MESSAGING_KNOTS)}, name="scoring")

MODELS = {"the model we expect (5 knots)": spline_spec((300.0, 750.0, 1300.0, 1900.0, 2500.0)),
          "a model with more bends (8 knots)": spline_spec(T.PLANNING_KNOTS)}

chosen = T.chosen_design()
candidates = {
    "two arms: 0 and 90 min": [(0.0, T.MESSAGING_MAX)] * 60 + [(1800.0, T.MESSAGING_MAX)] * 60,
    "five arms x 3 messaging": T.snap([(t, m) for t in (0.0, 600.0, 1200.0, 1800.0, 3000.0)
                                       for m in T.MESSAGING_LEVELS] * 8),
    "ten arms x 3 messaging": T.snap([(t, m) for t in np.linspace(0.0, T.TUTORING_MAX, 10)
                                      for m in T.MESSAGING_LEVELS] * 4),
    "D-optimal exchange": chosen,
}
grid = np.array([900.0, 1800.0, 3000.0])
print(f"{'design':26s} {'levels':>7} " + " ".join(f"{k.split('(')[1][:-1]:>18s}" for k in MODELS))
print(f"{'':26s} {'':>7} " + " ".join(f"{'logdet  se@45/90/150':>18s}" for _ in MODELS))
for name, points in candidates.items():
    cells = []
    for spec_ in MODELS.values():
        X = matrix(points, spec_)
        se = curve_se(points, spec_, grid)
        cells.append(f"{d_criterion(X):7.2f} {se[0]:.2f}/{se[1]:.2f}/{se[2]:.2f}"
                     if np.isfinite(d_criterion(X)) else f"{'singular':>18s}")
    print(f"{name:26s} {len(set(a for a, _ in points)):>7d} " + " ".join(f"{c:>18s}" for c in cells))

The five-arm factorial is the **most precise design there is** for the model the analysis
expects — about a tenth tighter than anything else at the ends of the range, three per
cent in the middle, and the best `d_criterion` in the column. It is also **singular** for a model with one more bend in it: five distinct doses
cannot fit eight knots, so a five-arm trial can report a curve and can never check whether
that curve was the right shape.

The two-arm trial is singular for both. It estimates one contrast, which is the question
notebook 1 showed was already answered.

`optimal_exchange` resolves this by being D-optimal against the *flexible* model: eleven
distinct tutoring levels, a tenth of the five-arm design's precision given up if the simple
model is right, and the best-conditioned design in the table if it is not. That is the
design the trial runs.

In [ ]:
fine = np.linspace(0.0, T.TUTORING_MAX, 61)
fig = T.figure("Precision the design buys, computed before any data exists",
               "tutoring", f"posterior sd of the fitted response ({T.OUTCOME_UNIT})", height=430)
for i, (name, points) in enumerate(candidates.items()):
    se = curve_se(points, MODELS["a model with more bends (8 knots)"], fine)
    if np.all(np.isnan(se)):
        continue
    fig.add_trace(go.Scatter(x=fine, y=se, name=name,
                             line={"color": ("#5b6472", "#c9a227", "#3aa17e", T.TUTORING_COLOR)[i],
                                   "width": 3,
                                   "dash": "solid" if name == "D-optimal exchange" else "dash"}))
fig.add_annotation(x=1800.0, y=0.9, text="the two-arm design is singular here:<br>no line to draw",
                   showarrow=False, font={"color": T.DECISION_COLOR})
T.minutes_axis(fig)
fig.show()

In [ ]:
allocation = pd.Series([f"{t:.0f}" for t, _ in chosen]).value_counts().sort_index(key=lambda s: s.astype(float))
fig = T.figure("Where the 120 schools go", "tutoring", "schools", height=380)
for level, colour in zip(T.MESSAGING_LEVELS, (T.TRUTH_COLOR, T.MESSAGING_COLOR, T.TUTORING_COLOR)):
    at_level = pd.Series([t for t, m in chosen if m == level]).value_counts().sort_index()
    fig.add_trace(go.Bar(x=at_level.index, y=at_level.to_numpy(),
                         name=f"{level / T.MESSAGE_COST:.0f} messages/wk", marker_color=colour))
fig.update_layout(barmode="stack")
T.minutes_axis(fig)
fig.show()
print("distinct tutoring levels:", sorted({t for t, _ in chosen}))
print("schools at zero tutoring:", sum(1 for t, _ in chosen if t == 0.0))

The exchange puts weight at the ends, at the middle, and — importantly — at 800 to 900
dollars, which is where notebook 1's arithmetic said the decision would sit. Fifteen
schools get no tutoring at all, which is the comparison the whole curve is anchored to and
the only genuine cost the trial imposes on its own students.

## 4 · How many schools

Now, and only now, the size question. Every candidate size gives an exact standard error
on the **decision quantity** — the difference in cohort points between funding 35 minutes
for everyone and 90 minutes for whoever the budget reaches — and `evoi_gaussian` turns
that standard error into money.

The program is expected to run for five years, so the information is worth five
discounted years of the annual EVPI. The trial's delivery is paid by a 10 million dollar
federal implementation grant, which is a **feasibility ceiling**, not a cost: those
students get tutored either way, and the state's own budget funds the other 120 schools
in the tier at the current best guess, so no child waits. What the state pays for out of
its own money is measurement.

In [ ]:
def decision_se(points, *, sigma=SIGMA):
    """Exact se of the decision quantity, from the design alone."""
    spec_ = MODELS["a model with more bends (8 knots)"]
    X = matrix(points, spec_).X
    covariance = np.linalg.inv(X.T @ X) * sigma**2
    zero = matrix([(0.0, 0.0)], spec_).X[0]
    low = matrix([(700.0, T.MESSAGING_MAX)], spec_).X[0] - zero
    high = matrix([(1800.0, T.MESSAGING_MAX)], spec_).X[0] - zero
    weight = (float(T.students_served(700.0, T.MESSAGING_MAX)) * low
              - float(T.students_served(1800.0, T.MESSAGING_MAX)) * high)
    return float(np.sqrt(weight @ covariance @ weight))

HORIZON, DISCOUNT = 5, 0.03
horizon_factor = mid_horizon_factor(HORIZON, DISCOUNT) * HORIZON
GRANT = 10_000_000.0
print(f"discount weights {np.round(discount_weights(HORIZON, DISCOUNT), 3)} -> "
      f"{horizon_factor:.2f} program-years of EVPI, so perfect information is worth "
      f"{evpi * horizon_factor:,.0f} USD")

sizes = (40, 60, 90, 120, 150, 180, 240)
plan = []
for n in sizes:
    points = T.chosen_design(n_schools=n)
    se = decision_se(points)
    value = evoi_gaussian(dose_decision, PRIOR_MEAN, PRIOR_SD, se)
    delivery = float(np.mean([t + m for t, m in points])) * 50.0 * n
    measurement = 6_500.0 * n + 120_000.0
    plan.append({"schools": n, "se": se, "EVSI/yr": value.evsi,
                 "EVSI over horizon": value.evsi * horizon_factor,
                 "share of EVPI": value.evsi / evpi, "measurement": measurement,
                 "net": value.evsi * horizon_factor - measurement,
                 "delivery": delivery, "fits the grant": delivery + measurement <= GRANT})
sizing = pd.DataFrame(plan).set_index("schools")
print()
print(sizing.round(2).to_string(formatters={c: "{:,.0f}".format for c in
      ("se", "EVSI/yr", "EVSI over horizon", "measurement", "net", "delivery")}))

In [ ]:
fig = T.figure("Sizing the trial: what another school is worth", "schools in the trial",
               "USD over the five-year program", height=420)
fig.add_trace(go.Scatter(x=sizing.index, y=sizing["EVSI over horizon"], name="value of the information",
                         line={"color": T.TUTORING_COLOR, "width": 3}))
fig.add_trace(go.Scatter(x=sizing.index, y=sizing["measurement"], name="what the state pays",
                         line={"color": T.DECISION_COLOR, "width": 2, "dash": "dash"}))
fig.add_trace(go.Scatter(x=sizing.index, y=sizing["net"], name="net", fill="tozeroy",
                         fillcolor=T.rgba(T.MESSAGING_COLOR, 0.18),
                         line={"color": T.MESSAGING_COLOR, "width": 2}))
fig.add_hline(y=evpi * horizon_factor, line={"dash": "dot", "color": T.TRUTH_COLOR},
              annotation_text="perfect information", annotation_position="top left")
largest = int(sizing.index[sizing["fits the grant"]].max())
fig.add_vline(x=largest, line={"color": T.ACCENT, "width": 2},
              annotation_text=f"{largest} schools — the largest the grant delivers")
fig.show()
row = sizing.loc[largest]
print(f"{largest} schools: se {row['se']:,.0f} points, {row['share of EVPI']:.0%} of perfect information, "
      f"net {row['net']:,.0f} USD over the horizon")
print(f"  MDE on the growth scale at {largest} schools: "
      f"{mde(largest, sd=float(table.loc['growth', 'school-level sd']), power=0.8).effect:.2f} {T.OUTCOME_UNIT}")
print(f"  reaching the 1.50 target exactly would take "
      f"{int(table.loc['growth', 'schools for 1.5 points'])} schools, which the grant will not deliver")
print(f"180 schools would net {sizing.loc[180, 'net'] - row['net']:,.0f} USD more and cost "
      f"{sizing.loc[180, 'delivery'] - row['delivery']:,.0f} USD of delivery nobody has")

## 5 · The analysis plan, written before the data

The design is fixed; so is what will be done with it. A `SurfaceSpec` is a value with a
content hash, which makes "the model we pre-registered" a checkable claim rather than a
recollection.

In [ ]:
registered = T.planning_spec()
print("pre-registered surface :", registered.name)
print("content hash           :", registered.content_hash()[:16])
print("response family        :", {k: v.name for k, v in registered.kernels.items()})
print("intercept              :", registered.intercept, "(a per-school intercept is collinear "
      "with a school-level dose)")
print("outcome                :", registered.outcome.name, f"({registered.outcome.description})")
print("unit of analysis       : one row per school (six benchmark readings averaged first)")
print("\ndecision rule: fund the allocation with the highest posterior mean cohort points,")
print("and report the whole posterior on the difference against the pilot's 90 minutes.")

## What notebook 2 decided

1. **Schools, not students.** Interference is real here, and it costs a factor of 4.3 in
   children at every effect size. On the raw gain scale the trial would need 580 schools
   and the state has 240.
2. **Growth, not gain.** Differencing against the same school's prior year removes the
   persistent between-school variance and takes the trial from 500 schools to 157. It is
   the single largest design decision in this notebook and there is no invoice for it.
3. **Eleven dose levels, not five.** The five-arm factorial is about a tenth more precise
   for the model we expect and *singular* for a model with one more bend.
   `optimal_exchange` against the flexible model gives up that tenth to keep the check —
   and notebook 3 is where that check earns its keep.
4. **120 schools**, because that is what the grant delivers. It buys a third of perfect
   information and nets 2.5 million dollars over the program's five years. It does *not*
   reach the 1.5-point target, which would take 157; the honest statement is that the
   trial is sized by the grant and lands at an MDE of 1.72, and the value-of-information
   table is what says that is an acceptable place to stop rather than a compromise nobody
   priced. The next 60 schools would add 0.9 million of value and 4.4 million of delivery
   cost that nobody has.

Notebook 3 runs it.